# 10 · Linear regression (ridge) and SVM

Two straight-line models on the same cleaned features.

**Needs:** `features2.parquet` (02). **Writes:** `ls_*` predictions, `submission_linear_regression.csv`, `submission_svm.csv`. **Time:** about 15 minutes.

- **Linear regression:** plain least squares on a 0/1 target overfits with hundreds of features, so we use *ridge* (a penalty that shrinks the coefficients). Best penalty was alpha = 30,000 → AUC **0.7440**. Its raw outputs can fall outside 0-1, so they are rescaled to probabilities with a ranking-preserving step.
- **SVM:** an exact RBF-kernel SVM is too slow for 77,000 rows, so we compared a linear SVM with a fast RBF approximation (Nystroem). Linear won: **0.7431** at C = 0.0005, versus 0.7344 for the RBF approximation.

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

In [ ]:
import pandas as pd, numpy as np, time, sys
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem

def main(which):
    D=DATA; W=WORK
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    X=pd.read_parquet(f'{W}/features2.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    T=X.merge(tr,on='encounter_id'); S=te.merge(X,on='encounter_id',how='left'); y=T.readmit_30d.values
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    cat=[c for c in feats if not pd.api.types.is_numeric_dtype(T[c])]; num=[c for c in feats if c not in cat and T[c].notna().sum()>0 and T[c].nunique()>1]
    A=pd.concat([T[num],S[num]]); miss=A.columns[A.isna().mean()>0.01]; lo,hi=T[num].quantile(.005),T[num].quantile(.995); med=T[num].clip(lo,hi,axis=1).median()
    prep=lambda df: pd.concat([df[num].clip(lo,hi,axis=1).fillna(med),df[miss].isna().astype(int).add_suffix('_na'),df[cat].astype(object).fillna('NA')],axis=1)
    Z=pd.get_dummies(pd.concat([prep(T),prep(S)],ignore_index=True),columns=cat,dtype=float).astype(np.float32); Zt,Zs=Z.iloc[:len(T)].values,Z.iloc[len(T):].values; print('matrix',Zt.shape,flush=True)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); t0=time.time()
    if which=='ridge': cfgs={f'ridge_a{a}':('ridge',a) for a in RIDGE_ALPHAS}
    else:
        cfgs={f'linsvc_C{c}':('lin',c) for c in SVC_C}; cfgs.update({f'rbf_g{g}_C0.005':('rbf',(g,0.005)) for g in RBF_GAMMAS})
    res={}
    for name,(kind,par) in cfgs.items():
        oof=np.zeros(len(T)); tst=np.zeros(len(S))
        for f,(a,b) in enumerate(folds):
            sc=StandardScaler().fit(Zt[a]); xa,xb,xs=[np.clip(sc.transform(M),-6,6).astype(np.float32) for M in (Zt[a],Zt[b],Zs)]
            if kind=='ridge': m=Ridge(alpha=par).fit(xa,y[a]); fa=m.predict
            elif kind=='lin': m=LinearSVC(C=par,dual=False,max_iter=2000).fit(xa,y[a]); fa=m.decision_function
            else:
                ny=Nystroem(gamma=par[0],n_components=1000,random_state=f).fit(xa[np.random.RandomState(f).choice(len(xa),20000,replace=False)])
                m=LinearSVC(C=par[1],dual=False,max_iter=2000).fit(ny.transform(xa),y[a]); fa=lambda M,m=m,ny=ny: m.decision_function(ny.transform(M))
            oof[b]=fa(xb); tst+=fa(xs)/5
        res[name]=roc_auc_score(y,oof); print(name,'OOF AUC %.5f'%res[name],'t=%.0f'%(time.time()-t0),flush=True)
        np.save(f'{W}/ls_{name}_oof.npy',oof); np.save(f'{W}/ls_{name}_test.npy',tst)
    print('BEST',max(res,key=res.get),max(res.values()))

In [ ]:
RIDGE_ALPHAS = [300, 3000, 30000, 100000]       # best: 30000
SVC_C        = [0.0001, 0.0005, 0.003]          # best: 0.0005
RBF_GAMMAS   = [0.0005, 0.002]                  # curved kernel: worse than linear
main('ridge'); main('svm')

### 10.1 Turn the scores into probabilities and write the two files
The rescaling is monotone, so the patient ranking (and AUC) is unchanged.

In [ ]:
import pandas as pd
def validate_submission(path):
    """Checks a file against sample_submission.csv: same ids in the same order, no gaps, probabilities in [0, 1]."""
    s = pd.read_csv(path); ss = pd.read_csv(f'{DATA}/sample_submission.csv')
    ok = s.encounter_id.tolist() == ss.encounter_id.tolist()
    print(Path(path).name, '| rows', len(s), '| ids match sample:', ok, '| NaN:', int(s.readmit_30d.isna().sum()),
          '| all in [0,1]:', bool(s.readmit_30d.between(0, 1).all()), '| mean %.4f' % s.readmit_30d.mean())
    assert ok and s.readmit_30d.notna().all() and s.readmit_30d.between(0, 1).all()

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
te = pd.read_csv(f'{DATA}/test.csv')
X = pd.read_parquet(f'{WORK}/features2.parquet', columns=['encounter_id']); y = X.merge(pd.read_csv(f'{DATA}/train.csv'), on='encounter_id').readmit_30d.values
for tag, fn in [('ridge_a30000', 'submission_linear_regression.csv'), ('linsvc_C0.0005', 'submission_svm.csv')]:
    o = np.load(f'{WORK}/ls_{tag}_oof.npy'); t = np.load(f'{WORK}/ls_{tag}_test.npy')
    m = LogisticRegression(C=1e6).fit(o.reshape(-1, 1), y); p = m.predict_proba(t.reshape(-1, 1))[:, 1]
    pd.DataFrame({'encounter_id': te.encounter_id, 'readmit_30d': np.round(p, 6)}).to_csv(f'{WORK}/{fn}', index=False)
    print(tag, 'CV AUC %.5f' % roc_auc_score(y, o)); validate_submission(f'{WORK}/{fn}')